# tracegraph 시작하기: 실습 가이드북

tracegraph를 처음 쓰는 분을 위한 소개 자료이자 실습서입니다. 위에서부터 셀을 차례로 실행하면
작은 LangGraph agent를 실제로 돌려 실행 기록을 만들고, 그 기록에서 실패 원인을 찾고, 정상 실행과
비교하는 과정을 각 명령의 실제 출력과 함께 확인할 수 있습니다.

**이 노트북을 마치면**

- tracegraph가 어떤 문제를 풀고, 이미 쓰고 있는 관찰 도구와 어떻게 함께 쓰는지 설명할 수 있습니다.
- 최종 결과에 가려진 실패를 찾고, 그 실패가 어디서 비롯됐는지 따라갈 수 있습니다.
- 각 명령의 출력이 무엇을 말해 주고 무엇을 말해 주지 않는지 구분하고, 다음에 할 일을 정할 수 있습니다.
- 변경 전후 실행을 비교하고, 그 비교를 CI나 테스트의 통과/실패 조건으로 바꿀 수 있습니다.
- 여러 실행의 진단 보고서를 모아 실패 현황을 집계할 수 있습니다.
- 내 agent에 적용할 때 무엇을 먼저 확인해야 하는지 압니다.

명령만 빠르게 훑어보려면 [`docs/USAGE_KO.md`](../docs/USAGE_KO.md)를 보세요. 이 노트북은 같은
흐름을 직접 실행하면서 "왜, 언제 쓰는지"까지 함께 설명합니다.

| 구간 | 내용 |
|---|---|
| 0 | 왜 tracegraph인가: 문제, 활용 상황, 핵심 개념, 입력 경로 |
| 1 | 준비 |
| 실습 1–3 | 기록 만들기 → artifact로 변환 → 바로 진단. **여기까지가 기본 과정입니다.** |
| 실습 4–8 | 더 알아보기: 구조 보기, step id 찾기, 원인 추적, 실행 비교, 여러 trace 검색 |
| 실무 응용 1–2 | 명령 묶어 쓰기: 변경 검증(재시도 로직 추가), 여러 실행의 실패 현황 집계 |
| 실습 9–10 | 심화: 원인이 여러 개인 구조(fan-in), 공유용 메모 내보내기 |
| 정리 | 명령 요약, 내 프로젝트에 적용하는 순서 |

각 실습의 출력 뒤에는 **해석** 블록이 있습니다. 출력이 *말해 주는 것*, *말해 주지 않는 것*, 그래서
*다음에 할 일*을 정리한 것입니다. tracegraph는 실행의 구조와 인과관계를 보여 주지만, 각 단계가 무엇을
입력받고 출력했는지는 보지 않습니다. 그 경계를 아는 것이 결과를 올바르게 쓰는 출발점입니다.

## 0. 왜 tracegraph인가

> 바로 실행해 보고 싶다면 **1. 준비**로 건너뛰어도 됩니다. 이 장은 실습 중에 돌아와서 읽어도 됩니다.

### 풀려는 문제: agent의 실패는 잘 보이지 않는다

agent 한 번의 실행에는 계획, LLM 호출, tool 호출, 분기, 재시도가 얽혀 있고, 같은 입력이라도
실행마다 경로가 달라질 수 있습니다. 그래서 디버깅이 어렵습니다.

1. **최종 답만으로는 중간 실패가 보이지 않습니다.** tool이 실패해도 오류 처리, 재시도, 대체
   경로(fallback)가 그럴듯한 답을 만들어 내면 겉으로는 성공한 실행처럼 보입니다.
2. **실패한 곳과 원인이 있는 곳이 다릅니다.** tool 호출이 실패한 이유는 몇 단계 앞의 계획이나
   입력에 있는 경우가 많습니다. 실패 지점에서 거꾸로 따라가야 원인이 보입니다.
3. **실행 한 번은 일화일 뿐입니다.** 같은 실패가 반복되는지, 어떤 변경 이후에 생겼는지는 여러 실행을
   함께 봐야 알 수 있습니다.

### tracegraph가 하는 일

Phoenix 같은 관찰 도구는 실행 기록(trace)을 모으고 화면으로 보여 주고 평가하는 일을 맡습니다.
tracegraph는 그 기록을 **인과 그래프(causal graph)**, 즉 "어떤 단계가 무엇 때문에 실행됐는가"의
그래프로 다시 읽어서 다음 세 질문에 답합니다.

| 질문 | 명령 | 결과 |
|---|---|---|
| 이 실패는 어디서 비롯됐나? | `explain` | 실패 step에서 기록된 원인을 거꾸로 모두 따라간 경로 |
| 정상 실행과 구조가 무엇이 달라졌나? | `diff` | 같다/다르다의 정확한 판정과 갈라진 위치 |
| 이 실패 패턴이 여러 실행 중 어디에 있나? | `query` | 이름 붙은 인과 패턴(preset)에 맞는 실행 목록 |

`analyze`는 이 셋을 묶어 한 번에 진단 보고서를 만듭니다.

tracegraph는 기록을 **만들지 않고 읽기만** 합니다. agent 코드에 계측 코드를 추가할 필요가 없고,
이미 쓰고 있는 LangGraph checkpoint, OpenTelemetry span, Phoenix 기록을 그대로 입력으로 씁니다.
관찰 도구를 대체하지 않고, 그 위에 원인 분석과 회귀 비교를 더하는 도구입니다.

### 이런 상황에서 씁니다

| 상황 | 알고 싶은 것 | 쓰는 명령 | 실습 |
|---|---|---|---|
| 사용자 응답은 정상인데 로그에 오류 흔적이 있다 | 숨은 실패가 있었나, 왜 났나 | `analyze`, `explain` | 3, 6 |
| 프롬프트·모델·tool을 바꿨다 | 기준 실행과 비교해 동작이 어떻게 바뀌었나 | `analyze --baseline`, `diff` | 3, 7 |
| PR마다 동작 회귀를 자동으로 막고 싶다 | 구조가 달라졌나, 금지한 패턴이 나왔나 | `diff`, `query`의 exit code | 7, 8 |
| 운영 trace가 수백 개 쌓였다 | 특정 실패 패턴이 어느 실행에서 나왔나 | `query` | 8 |
| 여러 결과를 합치는 단계(RAG, 병렬 agent)에서 문제가 났다 | 합쳐진 원인 중 무엇이 영향을 줬나 | `explain` (raw graph) | 9 |
| 장애 회고를 팀과 공유하고 싶다 | 원문(prompt, 오류 문자열) 없이 구조만 남긴 기록 | `export-incident-memo` | 10 |

### 핵심 개념

| 용어 | 뜻 |
|---|---|
| trace | 실행 한 번의 기록. LangGraph에서는 thread 하나, OpenTelemetry에서는 trace id 하나입니다. |
| checkpoint | LangGraph가 각 단계가 끝날 때 저장하는 그래프 상태. 대화 하나(thread)마다 이력이 쌓입니다. |
| span | OpenTelemetry에서 작업 한 구간(LLM 호출 한 번, tool 호출 한 번 등)의 기록. 부모 span을 가리킬 수 있습니다. |
| span link | 부모 관계와 별도로 다른 span을 가리키는 참조. 예: 답변을 쓰는 span이 "검색 span의 결과를 썼다"고 표시하는 것. |
| step | trace 안의 실행 단위. LangGraph 노드 한 번 실행, span 하나에 해당합니다. |
| artifact | tracegraph가 원래 기록을 읽어 만든 정규화된 JSON 파일. 이후 모든 분석의 입력입니다. |
| `CAUSED_BY` edge | "이 step은 저 step 때문에 실행됐다"는 인과 관계. 결과 step에서 원인 step으로 향합니다. |
| baseline | 비교 기준이 되는 실행. 보통 같은 입력으로 돌린 정상 실행입니다. |
| preset | 이름과 버전이 붙은 인과 패턴. 예: `tool-failure@v1` = 실패한 TOOL step. |

**설계 원칙: 인과관계를 지어내지 않는다.** 한 step의 원인은 여러 개일 수 있습니다. 예를 들어 답변을
쓰는 step은 계획 단계와 검색 결과 둘 다에 의존할 수 있습니다. 그런데 화면에 보기 좋은 트리는 부모를
하나만 그릴 수 있습니다. tracegraph는 두 계층을 따로 저장합니다.

| 계층 | edge | 뜻 | 읽는 명령 |
|---|---|---|---|
| raw causal graph | `CAUSED_BY` | 기록된 실제 원인을 **모두** 보존 | `explain`, `query` |
| derived tree | `TREE_PARENT` | 보기 쉽게 부모를 **하나만** 남긴 트리 | `inspect`, `diff` |

이 구분이 중요한 이유: 트리에서 생략된 원인이 진짜 원인이라면, 트리만 보고 고친 곳은 엉뚱한 곳이
됩니다. 그래서 트리로 접으면서 원인이 생략된 step에는 `lossy-projection` 표시가 붙고, 원인 분석은
항상 raw 계층에서 합니다. 실습 9에서 이 차이를 직접 봅니다.

### 기록이 들어오는 길

| 이미 쓰고 있는 것 | 명령 | 특징 |
|---|---|---|
| LangGraph + `SqliteSaver` | `ingest --sqlite` | 별도 계측 없이 checkpoint 이력을 그대로 읽습니다. **이 노트북의 주 경로입니다.** |
| OpenTelemetry / OpenInference 계측 | `ingest-otlp` | span link까지 읽으므로 원인이 여러 개인 구조(fan-in)를 보존합니다. 실습 9. |
| Arize Phoenix | `phoenix diagnose`, `ingest-phoenix` | 가장 빠른 시작점. prompt·출력 같은 원문을 제거한 artifact를 만듭니다. Phoenix export에는 span link가 없어서 추가 원인이 빠졌을 수 있다고(`parent_only`) 표시합니다. |

`ingest`는 현재 `SqliteSaver` DB를 읽습니다. 다른 checkpointer를 쓴다면 OpenTelemetry나 Phoenix 경로를
쓰세요.

### 이 노트북의 예제

```
plan → call_tool ──(error)──▶ handle_error → respond     # thread A: 입력 "boom"
                 └──(ok)──────────────────▶ respond     # thread B: 입력 "hello"
```

- **A**: `call_tool`이 실패하지만 `handle_error`가 처리해서 그래프는 끝까지 갑니다. 실무로 치면
  검색 API가 오류를 냈지만 대체 경로로 답을 만들어 사용자는 눈치채지 못한 경우입니다. 이런
  "복구된 실패"가 tracegraph로 찾기 좋은 대상입니다.
- **B**: 정상 경로입니다. A와 나란히 놓고 차이를 보여 주는 비교 대상으로 씁니다. 입력은 A와 다릅니다.
  같은 입력으로 변경 전후를 비교하는 방법은 실무 응용 1에서 봅니다.

## 1. 준비

이 노트북은 tracegraph **저장소 checkout**에서 실행합니다. 예제 agent가 쓰는 `langgraph`는 개발용
의존성이라 PyPI 패키지(`agent-tracegraph`)만 설치해서는 들어오지 않기 때문입니다. 내 프로젝트에서
tracegraph만 쓸 때는 `uv tool install agent-tracegraph`나 `pip install agent-tracegraph`로 충분합니다.

필요한 것: git, Python 3.12, [uv](https://docs.astral.sh/uv/).

```bash
git clone https://github.com/memtomem/tracegraph.git
cd tracegraph
uv sync
uv run --with jupyterlab jupyter lab examples/getting_started_ko.ipynb
```

**커널 확인.** 아래 셀은 이 노트북의 커널이 **이 저장소의 tracegraph(`src/`)와 `langgraph`를 쓸 수
있는지** 검사합니다. `uv sync`로 만든 `.venv`나 `uv run --with jupyterlab`이 만드는 임시 환경은
통과합니다. 다른 Python을 가리키는 커널이 골라져 있으면 `langgraph`를 찾지 못하거나 다른 버전의
tracegraph를 실행하게 되므로, 그때는 셀이 멈추고 해결 방법을 알려 줍니다. 해결 방법은 프로젝트 커널을 등록한 뒤
Jupyter의 커널 메뉴에서 그 커널을 고르는 것입니다.

```bash
uv run python -m ipykernel install --user --name tracegraph-tutorial
```

VS Code에서는 커널 선택에서 저장소 `.venv`의 Python을 고르면 됩니다.

In [1]:
import json
import os
import shlex
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

# examples/tiny_agent.py 가 있는 디렉터리가 나올 때까지 위로 올라가 저장소 루트를 찾습니다.
REPO = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "examples" / "tiny_agent.py").is_file()),
    None,
)
if REPO is None:
    raise RuntimeError("tracegraph 저장소 안에서 이 노트북을 여세요 (examples/tiny_agent.py 를 찾지 못했습니다).")


def shown(path) -> str:
    """출력에 개인 경로가 남지 않도록 저장소 기준 경로로 바꿔 보여 줍니다."""
    path = Path(path)
    return f"<repo>/{path.relative_to(REPO)}" if path.is_relative_to(REPO) else str(path)


KERNEL_HELP = (
    "이 커널은 이 저장소의 tracegraph 와 langgraph 를 쓰지 않습니다. 저장소 루트에서 다음을 실행한 뒤\n"
    "Jupyter 커널 메뉴에서 'tracegraph-tutorial' 커널을 고르세요.\n"
    "    uv sync\n"
    "    uv run python -m ipykernel install --user --name tracegraph-tutorial"
)
try:
    import langgraph  # noqa: F401  (예제 agent 실행에 필요)
    import tracegraph
except ImportError as exc:
    raise RuntimeError(f"{exc}\n\n{KERNEL_HELP}") from None
if not Path(tracegraph.__file__).resolve().is_relative_to((REPO / "src").resolve()):
    raise RuntimeError(f"이 커널의 tracegraph 는 {tracegraph.__file__} 입니다.\n\n{KERNEL_HELP}")

sys.path.insert(0, str(REPO / "examples"))  # tiny_agent, otlp_agent_trace 를 import 하기 위해
print("Python    :", shown(sys.executable))
print("tracegraph:", tracegraph.__version__, "from", shown(tracegraph.__file__))

Python    : <repo>/.venv/bin/python3
tracegraph: 0.4.0 from <repo>/src/tracegraph/__init__.py


실습에 쓸 파일은 모두 임시 작업 디렉터리 `WORK` 아래에 만듭니다. 저장소는 건드리지 않습니다.

- `WORK/trace.db`: agent가 남긴 LangGraph checkpoint DB
- `WORK/traces/`: tracegraph artifact(JSON). `query`가 이 디렉터리를 통째로 읽습니다.
- `WORK/raw/`: 변환 전 원본 export
- `WORK/out/`: 분석 보고서, 내보낸 파일, 비교용 임시 artifact

`tg(...)`는 `tracegraph` 명령을 `WORK`에서 실행하고 출력과 **exit code**를 함께 보여 주는 작은
도우미입니다. 터미널에서 칠 명령과 똑같은 명령을 실행합니다. exit code를 굳이 보여 주는 이유는,
tracegraph의 몇몇 명령은 exit code 자체가 답이고 CI가 바로 그 값을 읽기 때문입니다(실습 7, 8).
`expect=`는 그 셀에서 기대하는 exit code이며, 다르면 셀이 실패합니다.

In [2]:
WORK = Path(tempfile.mkdtemp(prefix="tracegraph-tutorial-"))
TRACES, RAW, OUT = WORK / "traces", WORK / "raw", WORK / "out"
for directory in (TRACES, RAW, OUT):
    directory.mkdir()

# 커널과 같은 venv 에 설치된 tracegraph 명령을 씁니다.
TRACEGRAPH = Path(sys.executable).with_name("tracegraph")
if not TRACEGRAPH.exists():
    TRACEGRAPH = Path(shutil.which("tracegraph"))

# 색 없는 일정한 폭의 텍스트로 출력하게 합니다 (노트북에 그대로 보이도록).
ENV = {k: v for k, v in os.environ.items() if k not in ("FORCE_COLOR", "PY_COLORS")}
ENV.update(TERM="dumb", COLUMNS="120")


def tg(*args, expect=0, quiet=False):
    """`tracegraph <args>` 를 WORK 에서 실행하고 출력과 exit code 를 보여 줍니다.

    quiet=True 이면 명령과 exit code 만 보여 줍니다 (같은 명령을 여러 번 돌릴 때).
    """
    words = [os.path.relpath(a, WORK) if isinstance(a, Path) else str(a) for a in args]
    print("$ tracegraph", shlex.join(words))
    result = subprocess.run(
        [str(TRACEGRAPH), *words], cwd=WORK, env=ENV, capture_output=True, text=True
    )
    if not quiet:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="", file=sys.stderr)
    print(f"[exit code {result.returncode}]")
    assert result.returncode == expect, f"exit code {expect} 를 기대했지만 {result.returncode} 였습니다"


print("WORK:", WORK)
print()
print("터미널에서 같은 명령을 따라 하려면:")
print(f"  cd {WORK}")
# `uv run --with jupyterlab` 로 띄우면 커널은 임시 환경에서 돌기 때문에, 터미널에는 uv sync 가
# 만든 저장소 .venv 를 안내합니다.
ACTIVATE = REPO / ".venv" / "bin" / "activate"
if not ACTIVATE.exists():
    ACTIVATE = Path(sys.executable).with_name("activate")
print(f"  source {shown(ACTIVATE)}")

WORK: /var/folders/4n/mt3km5rs1bn_b0w_d6hvlmhc0000gn/T/tracegraph-tutorial-siaei0kp

터미널에서 같은 명령을 따라 하려면:
  cd /var/folders/4n/mt3km5rs1bn_b0w_d6hvlmhc0000gn/T/tracegraph-tutorial-siaei0kp
  source <repo>/.venv/bin/activate


In [3]:
tg("--version")

$ tracegraph --version
0.4.0
[exit code 0]


## 실습 1: 실행 기록 만들기

> **왜 checkpoint인가요?** LangGraph의 checkpointer는 원래 대화 이어 가기, 사람 승인을 기다렸다
> 재개하기(human-in-the-loop), 이전 상태로 되돌아가기 같은 기능을 위해 쓰는 저장소입니다. 단계가
> 끝날 때마다 상태를 저장하므로, 이미 checkpointer를 쓰고 있다면 그 이력이 곧 실행 기록입니다.
> 추가 계측이 필요 없습니다.

`examples/tiny_agent.py`의 그래프를 SQLite checkpointer와 함께 실행합니다. agent 코드에는 tracegraph
관련 코드가 한 줄도 없다는 점을 확인해 보세요. 실제 서비스에서는 thread 하나(대화나 작업 하나)가
tracegraph의 trace 하나가 됩니다.

In [4]:
from langgraph.checkpoint.sqlite import SqliteSaver

import tiny_agent

TRACE_DB = WORK / "trace.db"
with SqliteSaver.from_conn_string(str(TRACE_DB)) as saver:
    print("thread A:", tiny_agent.run(saver, "A", "boom"))
    print("thread B:", tiny_agent.run(saver, "B", "hello"))

thread A: {'input': 'boom', 'error': "tool failed on input 'boom'", 'result': 'recovered', 'log': ['planned', 'tool_error', 'handled', 'responded']}
thread B: {'input': 'hello', 'error': None, 'result': 'tool ok', 'log': ['planned', 'tool_ok', 'responded']}


A의 최종 `result`는 `'recovered'`입니다. 사용자에게 돌아간 답만 보면 문제가 없어 보이지만, 중간
`log`에는 `tool_error`가 남아 있습니다. 실제 agent에서는 이런 흔적이 수많은 로그 사이에 묻혀
지나가기 쉽습니다. 이제 이 실패를 tracegraph로 찾아봅니다.

## 실습 2: artifact로 변환 (`ingest`, `validate`)

> **왜 변환하나요?** artifact는 실행 기록을 tracegraph의 정규화된 형식으로 담은 JSON 파일입니다.
> - **재현과 공유**: 원래 DB나 서버 없이 파일 하나로 같은 분석을 다시 할 수 있습니다. 이슈에 첨부하거나
>   CI 입력으로 쓰기 좋습니다.
> - **입력 통일**: LangGraph, OpenTelemetry, Phoenix 어디서 왔든 같은 명령으로 분석합니다.
> - **원본 기록(system of record)**: 이후 모든 명령은 이 파일만 읽습니다. 형식은 `schema_version`으로
>   버전 관리됩니다.

In [5]:
A_JSON, B_JSON = TRACES / "A.json", TRACES / "B.json"
tg("ingest", "--sqlite", TRACE_DB, "--thread", "A", "--out", A_JSON)
tg("ingest", "--sqlite", TRACE_DB, "--thread", "B", "--out", B_JSON)

$ tracegraph ingest --sqlite trace.db --thread A --out traces/A.json
ingested 6 steps → traces/A.json
[exit code 0]
$ tracegraph ingest --sqlite trace.db --thread B --out traces/B.json


ingested 5 steps → traces/B.json
[exit code 0]


`validate`는 artifact가 tracegraph가 만든 그대로의 형태인지 검사합니다. CI에서 분석 전에 한 번
돌려 두면, 손으로 고친 파일이나 다른 도구가 만든 깨진 파일을 분석 단계에 들어가기 전에 걸러 낼 수
있습니다.

In [6]:
tg("validate", A_JSON, B_JSON)

$ tracegraph validate traces/A.json traces/B.json
OK traces/A.json  A · 6 steps
OK traces/B.json  B · 5 steps

2 artifact(s) valid
[exit code 0]


## 실습 3: 바로 진단 (`analyze`)

> **언제 쓰나요?** 처음 보는 실패 trace를 받았을 때 가장 먼저 실행하는 명령입니다. step id 같은 세부
> 사항을 몰라도 실패 후보, 원인 경로, 반복 패턴, 분석 한계 안내를 한 번에 보여 줍니다. Phoenix를 쓴다면
> `tracegraph phoenix diagnose`가 같은 보고서를 Phoenix에서 바로 만듭니다.

In [7]:
tg("analyze", A_JSON)

$ tracegraph analyze traces/A.json


A  6 steps · 1 error

What failed — candidates to investigate
  ✗ call_tool (TOOL, step 2)
    CHAIN → CHAIN [checkpoint_parent]
    CHAIN → plan [checkpoint_parent]
    plan → call_tool [checkpoint_parent]

Repeated behavior
  • plan-then-tool-failure@v1: plan → call_tool
  • tool-failure@v1: call_tool
  • error@v1: call_tool

Telemetry
  wall=unavailable · tokens=unavailable · cost=unavailable

Data fidelity / privacy
  declared_dag · links_preserved=None · privacy=safe-v1
  ⚠ current: LangGraph failure capture covers the configured error channel and task pending writes for recognized 
checkpoint layouts; absence of an observed failure does not prove execution success.
  ⚠ Failure candidates are investigation leads. Containment/unknown edges do not prove propagation; explicit causal 
ancestry is context, not proof of exception propagation.
[exit code 0]


출력 읽는 법:

- **범례**: `call_tool (TOOL, step 2)`에서 `TOOL`은 step의 종류(tool 호출)이고 `step 2`는 기록 순서(`seq`)
  입니다. `CHAIN`은 tool이 아닌 일반 단계입니다. 이름 없이 `CHAIN`으로만 나오는 두 step은 그래프 입력
  (`seq` -1)과 첫 루프 시작(`seq` 0)을 기록한 LangGraph checkpoint입니다. `A → B`는 "A 다음에 B가
  실행됐다"는 인과 관계입니다.
- **What failed**: 실패한 step(`call_tool`, 종류 `TOOL`)과 그 앞의 인과 경로입니다.
  `[checkpoint_parent]`는 그 edge가 어디서 왔는지(LangGraph checkpoint의 부모 관계)를 뜻합니다.
  최종 결과가 `recovered`였던 실행에서 숨은 실패를 찾아냈습니다.
- **Repeated behavior**: 이 trace에서 걸린 preset 패턴입니다. 실습 8에서 다시 봅니다.
- **Telemetry**: 시간·token·비용입니다. 이 예제 agent는 기록하지 않으므로 `unavailable`입니다.
  값이 없을 때 0으로 채우지 않고 `unavailable`이라고 쓰는 것도 "지어내지 않는다" 원칙입니다.
- **Data fidelity / privacy**: 이 분석을 어디까지 믿을 수 있는지에 대한 분석 한계 안내(disclosure)입니다.
  - `declared_dag`: 기록을 만든 쪽이 선언한 인과관계(여기서는 checkpoint의 부모 관계)가 빠짐없이
    보존됐다는 뜻입니다. Phoenix에서 가져온 기록은 `parent_only`로 표시되며, 추가 원인이 빠졌을 수
    있다는 뜻입니다.
  - `links_preserved=None`: LangGraph 기록에는 span link라는 개념이 없어서 "해당 없음"입니다. "링크가
    없다"는 확정으로 읽으면 안 됩니다. OpenTelemetry 기록에서는 `True`/`False`로 나옵니다.
  - `privacy=safe-v1`: 보고서가 따르는 본문 제거 규칙입니다. 아래 "JSON 보고서 활용"에서 설명합니다.
  - 경고 두 줄은 "실패 후보는 조사할 단서이지 증명이 아니다", "실패가 관찰되지 않았다고 성공이 증명되는
    것은 아니다"라는 뜻입니다. 두 번째는 이 장 끝에서 직접 확인합니다.

> **해석**
> - *말해 주는 것*: 이 실행에서 실패가 기록된 step은 `call_tool` 하나이고, 그 앞의 인과 경로는
>   입력 → 루프 시작 → `plan`입니다. 사용자 응답이 `recovered`였어도 실패는 있었습니다.
> - *말해 주지 않는 것*: tool이 **왜** 실패했는지, 그 실패가 답변 품질에 영향을 줬는지는 알려 주지
>   않습니다. tracegraph는 실행의 구조와 인과관계를 볼 뿐, tool 내부나 응답 내용은 보지 않습니다.
> - *다음에 할 일*: 원인 경로 위의 `plan`이 tool에 무엇을 넘겼는지 원래 기록(Phoenix 화면, 로그,
>   checkpoint state)에서 확인합니다. 같은 실패가 다른 실행에서도 반복되는지는 `query`로 봅니다(실습 8).

**정상 실행과 비교하기.** "무엇이 실패했나"만큼 자주 묻는 질문이 "평소와 무엇이 달랐나"입니다.
정상 실행을 `--baseline`으로 넘기면 차이를 함께 보여 줍니다. 실무에서는 **같은 입력**으로 돌린 정상
실행을 기준으로 삼습니다. 여기서는 차이가 잘 드러나도록 입력이 다른 B(`"hello"`)를 쓰고, 같은 입력
비교는 실무 응용 1에서 합니다. `--json-out`은
같은 내용을 기계가 읽는 JSON 보고서로 저장합니다.

In [8]:
tg("analyze", A_JSON, "--baseline", B_JSON, "--json-out", OUT / "analysis.json")

$ tracegraph analyze traces/A.json --baseline traces/B.json --json-out out/analysis.json


A  6 steps · 1 error

What failed — candidates to investigate
  ✗ call_tool (TOOL, step 2)
    CHAIN → CHAIN [checkpoint_parent]
    CHAIN → plan [checkpoint_parent]
    plan → call_tool [checkpoint_parent]

Repeated behavior
  • plan-then-tool-failure@v1: plan → call_tool
  • tool-failure@v1: call_tool
  • error@v1: call_tool

Telemetry
  wall=unavailable · tokens=unavailable · cost=unavailable

Compared with baseline
  topology: changed
  • call_tool: ok → error
  • respond: ok → missing
  • handle_error: missing → ok
  • respond: missing → ok
  pattern deltas: error=+1, plan-then-tool-failure=+1, tool-failure=+1
  telemetry delta: wall=unavailable · tokens=unavailable · cost=unavailable

Data fidelity / privacy
  declared_dag · links_preserved=None · privacy=safe-v1
  ⚠ current: LangGraph failure capture covers the configured error channel and task pending writes for recognized 
checkpoint layouts; absence of an observed failure does not prove execution success.
  ⚠ baseline: LangGr

**Compared with baseline** 읽는 법:

- `topology: changed`: 실행 구조가 다릅니다.
- `call_tool: ok → error`: 같은 자리에 있는 step의 상태가 바뀌었습니다.
- step은 트리 안의 **위치**로 짝지어집니다. A에서는 `respond`가 `handle_error` 아래로 자리를
  옮겼기 때문에, B 쪽 자리에서는 `ok → missing`, A 쪽 새 자리에서는 `missing → ok`로 한 번씩
  나타납니다.
- `pattern deltas`: baseline보다 늘어난 패턴 수입니다. "변경 후 tool 실패가 1건 늘었다"처럼 읽습니다.

차이가 있어도 `analyze`의 exit code는 0입니다. `analyze`는 사람이 읽는 진단이고, CI에서 "달라지면
실패"를 판정할 때는 `diff`와 `query`를 씁니다(실습 7, 8).

**JSON 보고서 활용.** 저장된 보고서는
[`contracts/analysis-report.schema.json`](../contracts/analysis-report.schema.json) 스키마를 따르는
버전 관리된 형식입니다. prompt, 출력, 원문 오류 메시지 같은 **본문**은 담지 않습니다(`privacy=safe-v1`).

다만 **익명화는 아닙니다.** trace id, step id, 그리고 식별자 형태(영문·숫자·`_.:/#@-`로만 된 문자열)의
step 이름은 그대로 남습니다. 예를 들어 `alice@example.com` 같은 값이 노드 이름이나 trace id에 쓰였다면
보고서에도 남습니다. 공백이 섞인 자유 문장 같은 이름은 `redacted:<해시>`로 바뀝니다. 그래서 보고서를
이슈나 대시보드에 넘기기 전에, trace id나 노드 이름에 사용자를 식별할 수 있는 값이 들어가지 않는지
확인하세요. 이름까지 감춰야 한다면 별칭과 digest만 남기는 실습 10의 메모를 씁니다.

In [9]:
report_text = (OUT / "analysis.json").read_text()
report = json.loads(report_text)
print("status      :", report["status"], "| steps:", report["step_count"], "| errors:", report["error_count"])
failure = report["primary_failures"][0]["step"]
print("failure     :", failure["name"], f"({failure['kind']}, step {failure['seq']})")
print("topology    :", "identical" if report["comparison"]["topology_identical"] else "changed")

# 확인: 오류 원문은 artifact 에는 있지만 보고서에는 없습니다.
assert "boom" in A_JSON.read_text() and "boom" not in report_text
print("원문 오류 문자열 'boom': artifact 에 있음, 보고서에 없음")

status      : error | steps: 6 | errors: 1
failure     : call_tool (TOOL, step 2)
topology    : changed
원문 오류 문자열 'boom': artifact 에 있음, 보고서에 없음


### 내 agent에 적용하기 전에: 실패가 제대로 잡히는지 확인하기

tracegraph는 LangGraph 기록에서 두 가지를 실패로 읽습니다. 하나는 노드가 지정한 state 키
(`--error-channel`, 기본값 `error`)에 값을 쓴 경우이고, 다른 하나는 노드가 예외로 끝나 LangGraph가
그 오류를 기록한 경우입니다. 예제의 `call_tool`은 `error` 키에 오류를 씁니다.

내 agent가 오류를 다른 키에 기록한다면 어떻게 될까요? 같은 thread A를 존재하지 않는 키
`failure`로 읽어 보겠습니다. 이 artifact는 `query`에 섞이지 않도록 `out/`에 둡니다. 결과는 실행 흐름을
트리로 보여 주는 `inspect`로 확인합니다. 트리 읽는 법은 실습 4에서 자세히 봅니다.

In [10]:
WRONG = OUT / "A-wrong-channel.json"
tg("ingest", "--sqlite", TRACE_DB, "--thread", "A", "--error-channel", "failure", "--out", WRONG)
tg("inspect", WRONG)

$ tracegraph ingest --sqlite trace.db --thread A --error-channel failure --out out/A-wrong-channel.json


ingested 6 steps → out/A-wrong-channel.json
[exit code 0]
$ tracegraph inspect out/A-wrong-channel.json


A (langgraph)
└── -1: CHAIN
    └── 0: CHAIN
        └── 1: plan
            └── 2: call_tool
                └── 3: handle_error
                    └── 4: respond

6 steps · 1 tool · 0 error · 0 lossy-projection
[exit code 0]


같은 실행인데 `0 error`가 나오고, 명령은 모두 성공(exit code 0)합니다. 트리에는 `handle_error`로
간 흔적이 그대로 있는데도 실패 표시만 사라졌습니다. **"실패가 보이지 않는다"는
"실패가 없었다"는 뜻이 아닙니다.** 그래서 tracegraph의 진단 보고서에는 이런 한계를 알려 주는 분석 한계
안내가 항상 붙습니다(위 `analyze` 출력의 *Data fidelity*).

내 agent에 처음 적용할 때는 다음 두 가지를 확인하세요.

- **실패 기록 방식**: 오류를 state에 쓴다면 그 키를 `--error-channel`로 알려 줍니다. 실패가 확실한
  실행 하나를 골라 `analyze`로 그 실패가 잡히는지 먼저 확인하는 것이 가장 빠른 점검입니다.
- **tool 노드 이름**: 노드 이름에 `tool` 또는 `tools`가 하나의 단어로 들어 있어야(`tools`,
  `call_tool`, `ToolNode`) step 종류가 `TOOL`로 분류되고 `tool-failure` 같은 preset이 적용됩니다.
  `toolnode`처럼 붙여 쓴 이름은 `TOOL`로 분류되지 않습니다.

**민감 정보 주의**: LangGraph와 OpenTelemetry에서 만든 artifact에는 step 이름과 오류 문자열이 원문
그대로 들어갑니다. 실제로 `A.json`에는 오류 메시지 `tool failed on input 'boom'`이 그대로 들어
있습니다. 오류 문자열에 사용자 입력이나 비밀 값이 섞일 수 있다면 artifact를 민감 자료로 다루세요.
위의 JSON 보고서는 이런 본문을 빼고, 실습 10의 메모는 이름까지 별칭으로 바꿉니다. 둘의 차이는 위의
"JSON 보고서 활용"에서 설명했습니다.

**기본 과정은 여기까지입니다.** 실행 기록을 만들고(실습 1), artifact로 바꾸고(실습 2), 숨은 실패와 그
원인 경로, 정상 실행과의 차이를 확인한 뒤 내 agent에서 실패가 잡히는지 점검하는 법을 봤습니다(실습 3).

아래부터는 **더 알아보기**입니다. 각 명령을 따로 쓰는 법과 실무에서 쓰는 방식, tracegraph가
인과관계를 다루는 방식을 조금 더 깊이 봅니다.

---

## 실습 4 (더 알아보기): 구조 보기 (`inspect`)

> **언제 쓰나요?** 실행이 어떤 분기를 탔는지 한눈에 보고 싶을 때, 또는 코드 리뷰나 장애 보고에서 실행
> 흐름을 설명할 때 씁니다.

각 줄은 `순서(seq): 이름`이고, 실패한 step에는 `✗`와 오류 메시지가 붙습니다. 마지막 줄은
step·tool·error·lossy-projection 개수 요약입니다.

In [11]:
tg("inspect", A_JSON)

$ tracegraph inspect traces/A.json


A (langgraph)
└── -1: CHAIN
    └── 0: CHAIN
        └── 1: plan
            └── 2: call_tool  ✗ tool failed on input 'boom'
                └── 3: handle_error
                    └── 4: respond

6 steps · 1 tool · 1 error · 0 lossy-projection
[exit code 0]


In [12]:
tg("inspect", B_JSON)

$ tracegraph inspect traces/B.json


B (langgraph)
└── -1: CHAIN
    └── 0: CHAIN
        └── 1: plan
            └── 2: call_tool
                └── 3: respond

5 steps · 1 tool · 0 error · 0 lossy-projection
[exit code 0]


두 트리를 나란히 보면 A만 `call_tool` 아래에서 `handle_error`로 갈라진 것이 보입니다. 이 차이를
사람이 눈으로 찾는 대신 정확히 판정하는 것이 실습 7의 `diff`입니다.

이 트리는 derived tree(`TREE_PARENT`)입니다. 이 예제처럼 한 줄로 이어진 실행에서는 raw graph와
모양이 같지만, 원인이 여러 개인 step이 있으면 달라집니다(실습 9). `seq`가 -1과 0인 이름 없는
`CHAIN` 두 개는 그래프 입력과 첫 루프 시작을 기록한 LangGraph checkpoint입니다.

## 실습 5 (더 알아보기): step id 찾기

> **언제 필요한가요?** 실패 후보는 `analyze`가 알려 주므로 id를 직접 찾을 일은 많지 않습니다. 다만
> 실패가 아닌 step을 포함해 **특정 step에서 출발해** 원인을 따라가고 싶을 때는 `explain`에 그 step의
> id를 넘겨야 합니다.

`inspect`는 id를 보여 주지 않습니다. id는 artifact JSON에 있고, artifact는 평범한 JSON이라 그대로
읽어 볼 수 있습니다.

In [13]:
artifact = json.loads(A_JSON.read_text())
print("top-level :", sorted(artifact))
print("trace     :", sorted(artifact["trace"]))
print()
steps = artifact["trace"]["steps"]
for step in steps:
    print(f"{step['seq']:>3}  {step['step_id']}  {str(step['name']):<13} {step['kind']:<6} {step['status']}")

top-level : ['schema_version', 'trace']
trace     : ['edges', 'steps', 'trace']

 -1  1f1c7bbb-6585-65e8-bfff-164ab99ccd65  None          CHAIN  ok
  0  1f1c7bbb-6586-6628-8000-d87a9e510981  None          CHAIN  ok
  1  1f1c7bbb-6587-644c-8001-3f5af0fa08fe  plan          CHAIN  ok
  2  1f1c7bbb-6587-6ed8-8002-64da2f4c6155  call_tool     TOOL   error
  3  1f1c7bbb-6588-689c-8003-e12ae54479e6  handle_error  CHAIN  ok
  4  1f1c7bbb-6588-6f90-8004-c1c0a0248935  respond       CHAIN  ok


LangGraph trace에서 step id는 checkpoint id라서 **실행할 때마다 바뀝니다.** 그래서 이 노트북은 id를
적어 두지 않고 매번 `status == "error"`인 step을 찾습니다. 스크립트에서 tracegraph를 쓸 때도 같은
방식이 안전합니다.

`explain`에는 전체 id 대신 **다른 step과 겹치지 않는 끝부분(suffix)**만 넣어도 됩니다. 터미널에서 긴
id를 복사하기 번거로울 때 편리합니다. 아래 함수는 겹치지 않는 가장 짧은 suffix를 구합니다.

In [14]:
errors = [s for s in steps if s["status"] == "error"]
# 확인: A 에는 실패 step 이 정확히 하나 있고, 그것은 TOOL 인 call_tool 입니다.
assert [(s["name"], s["kind"]) for s in errors] == [("call_tool", "TOOL")]
FAILED_ID = errors[0]["step_id"]


def unique_suffix(step_id, all_ids, minimum=4):
    for length in range(minimum, len(step_id) + 1):
        tail = step_id[-length:]
        if sum(other.endswith(tail) for other in all_ids) == 1:
            return tail
    return step_id


FAILED_SUFFIX = unique_suffix(FAILED_ID, [s["step_id"] for s in steps])
print("failed step id:", FAILED_ID)
print("unique suffix :", FAILED_SUFFIX)

# 확인: 정상 실행 B 에는 실패 step 이 없습니다.
assert not [s for s in json.loads(B_JSON.read_text())["trace"]["steps"] if s["status"] == "error"]

failed step id: 1f1c7bbb-6587-6ed8-8002-64da2f4c6155
unique suffix : 6155


## 실습 6 (더 알아보기): 원인 추적 (`explain`)

> **언제 쓰나요?** "이 tool 호출은 어떤 계획과 입력에서 나왔나?"를 알고 싶을 때입니다. 실패한 곳과
> 원인이 있는 곳은 다른 경우가 많아서, 실패 지점에서 거꾸로 따라가야 고칠 곳이 보입니다. 실패가 아닌
> step에도 쓸 수 있습니다. 예를 들어 "이 비싼 LLM 호출은 왜 일어났나?"를 따라갈 수 있습니다.

`explain`은 주어진 step에서 시작해 raw causal graph(`CAUSED_BY`)를 거꾸로 따라가며 기록된 원인을
모두 보여 줍니다. `←`는 "이전 원인"을 뜻합니다.

In [15]:
tg("explain", A_JSON, FAILED_ID)

$ tracegraph explain traces/A.json 1f1c7bbb-6587-6ed8-8002-64da2f4c6155


call_tool (step 2) [error] tool failed on input 'boom'
  ← plan (step 1)
  ← CHAIN (step 0)
  ← CHAIN (step -1)
[exit code 0]


In [16]:
tg("explain", A_JSON, FAILED_SUFFIX)  # suffix 만 넣어도 같은 결과

$ tracegraph explain traces/A.json 6155


call_tool (step 2) [error] tool failed on input 'boom'
  ← plan (step 1)
  ← CHAIN (step 0)
  ← CHAIN (step -1)
[exit code 0]


`call_tool` 실패는 `plan` → 루프 시작 → 입력으로 이어진 경로 위에 있습니다. 실제 agent라면 이
경로에서 "계획 단계가 tool에 어떤 인자를 넘기기로 했나"를 확인하는 것이 다음 조사 단계가 됩니다.
실패 뒤에 실행된 `handle_error`, `respond`는 원인이 아니라 결과이므로 나오지 않습니다.

> **해석**
> - *말해 주는 것*: 실패에 이르기까지 기록된 모든 앞 단계, 즉 **조사 범위**입니다. 이 범위 밖의
>   단계(여기서는 `handle_error`, `respond`)는 이 실패의 원인 후보에서 빼도 됩니다.
> - *말해 주지 않는 것*: 경로 위의 어느 단계가 **잘못했는지**는 알려 주지 않습니다. 경로에 있다는
>   것은 "이 단계 다음에 실행됐다"는 기록이지, "이 단계가 실패를 일으켰다"는 증명이 아닙니다.
> - *다음에 할 일*: 경로를 실패 지점에서 가까운 쪽부터 거슬러 올라가며 각 단계의 입력과 출력을 원래
>   기록에서 확인합니다. 실제 agent에서는 이 경로가 수십 단계일 수 있으므로, 범위를 좁혀 주는 것만으로도
>   조사 시간이 크게 줄어듭니다.

## 실습 7 (더 알아보기): 두 실행 비교 (`diff`)

> **언제 쓰나요?** 프롬프트, 모델, tool, 그래프 코드를 바꾼 뒤 같은 입력으로 다시 돌려서 실행 구조가
> 바뀌었는지 확인할 때입니다. 결과가 정확한 같다/다르다 판정이고 exit code로 나오므로 CI의 회귀
> 검사로 쓰기 좋습니다.

`diff`는 두 실행의 **derived tree**가 같은지, 즉 트리 모양과 각 step의 label(이름, 이름이 없으면
종류)이 같은지 비교합니다. 상태(`ok`/`error`)나 시간처럼 label이 아닌 값은 비교하지 않습니다.
같으면 exit code 0, 다르면 **1**입니다. 여기서 1은 오류가 아니라 "다르다"는 답입니다.

In [17]:
tg("diff", A_JSON, B_JSON, expect=1)

$ tracegraph diff traces/A.json traces/B.json


Derived TREE_PARENT comparison; lossy steps: A=0, B=0
NOT IDENTICAL
  • only in A under CHAIN > CHAIN > plan > call_tool: subtree handle_error(respond)
  • only in B under CHAIN > CHAIN > plan > call_tool: subtree respond
(divergence localization is heuristic; the identical/not verdict is exact)
[exit code 1]


`NOT IDENTICAL` 아래 줄은 어디서 갈라졌는지 보여 줍니다. `call_tool` 아래가 A에서는
`handle_error(respond)`, B에서는 `respond`입니다. 같다/다르다 판정은 정확하고, 갈라진 위치 표시는
안내용(heuristic)입니다.

> **해석**
> - *말해 주는 것*: 두 실행의 구조가 다르고(정확한 판정), `call_tool` 다음에서 갈라졌습니다(안내).
> - *말해 주지 않는 것*: 그 차이가 **좋은지 나쁜지**는 알려 주지 않습니다. A가 `handle_error`로 간 것은
>   설계한 복구 경로가 제대로 동작한 결과일 수도 있습니다. 또 label이 아닌 값(상태, 시간)의 차이는
>   비교하지 않으므로, `diff`가 같다고 해도 한쪽에서만 실패가 났을 수 있습니다. 상태 차이는
>   `analyze --baseline`이 보여 줍니다.
> - *다음에 할 일*: 의도한 변경이면 기준 실행(baseline)을 새 실행으로 바꿔 둡니다. 의도하지 않은
>   변경이면 갈라진 지점의 분기 조건(이 예제에서는 `call_tool` 뒤의 라우팅)을 확인합니다.

`--structure`는 이름(label)을 무시하고 모양만 비교합니다. 노드 이름을 바꾸는 리팩터링 뒤에도 실행
구조가 그대로인지 확인할 때 씁니다.

In [18]:
tg("diff", "--structure", A_JSON, B_JSON, expect=1)

$ tracegraph diff --structure traces/A.json traces/B.json


Derived TREE_PARENT comparison; lossy steps: A=0, B=0
NOT IDENTICAL
  • only in A: root subtree CHAIN(CHAIN(plan(call_tool(handle_error(respond)))))
  • only in B: root subtree CHAIN(CHAIN(plan(call_tool(respond))))
(divergence localization is heuristic; the identical/not verdict is exact)
[exit code 1]


**CI에서 쓰는 예.** 기준 실행(baseline)의 artifact를 저장소에 두고, 변경 후 같은 입력으로 만든
artifact와 비교합니다. exit code가 0이 아니면 CI 단계가 실패합니다. 파일이 없거나 인자가 잘못된
경우는 exit code 2이므로 "구조가 다름"(1)과 구분됩니다.

```bash
tracegraph diff baselines/checkout-flow.json artifacts/checkout-flow.json
```

LLM 응답에 따라 정상적으로도 경로가 달라지는 agent라면 모든 구조 차이를 실패로 볼 수는 없습니다.
그런 경우에는 "금지한 패턴이 나오지 않았다"를 검사하는 `query`가 더 알맞습니다(실습 8).

## 실습 8 (더 알아보기): 여러 trace 검색 (`presets`, `query`)

> **언제 쓰나요?** 실행 기록이 여러 개 쌓였을 때 "tool 실패가 어느 실행에서 났나", "같은 tool을
> 재시도했는데 또 실패한 경우가 있나" 같은 질문을 한 번에 던질 때입니다. 하나씩 열어 보는 대신
> 인과 패턴으로 검색합니다.

`query`는 이름 붙은 인과 패턴(preset)을 여러 artifact에서 찾습니다. 먼저 어떤 preset이 있는지
봅니다. `@v1`, `@v2`는 패턴 버전으로, 매칭 의미가 바뀌면 버전이 올라가므로 저장해 둔 결과와 나중에
비교할 수 있습니다.

In [19]:
tg("presets")

$ tracegraph presets


error@v1  any step that errored
    {status=error}
tool-failure@v1  a tool step that errored  review-exportable
    {kind=TOOL, status=error}
plan-then-tool-failure@v1  a 'plan' step immediately followed (causally) by a failing tool  review-exportable
    {name=plan} → {kind=TOOL, status=error}
timeout@v1  a step that failed with a bounded timeout status message
    {error~=timeout, status=error}
repeated-agent-failure@v1  the same agent failed again later on the causal path
    {kind=AGENT, status=error} →[gap 1..]→ {kind=AGENT, status=error, name=#0}
gate-failure-after-success@v1  a successful agent attempt immediately followed by a failing gate
    {kind=AGENT, status=ok} → {name^=gate:, kind=TOOL, status=error}
tool-retry-failure@v2  an explicitly linked retry of the same tool that fails  review-exportable
    {kind=TOOL} → {name^=retry:, kind=CHAIN} → {kind=TOOL, status=error, name=#0}
tool-repeat-failure-heuristic@v1  the same tool appears again later and fails; retry causality i

목록 읽는 팁:

- `→`는 바로 이어진 인과 단계를 뜻합니다. `→[gap 2..30]→`처럼 gap이 붙으면 그 사이에 다른 단계가
  끼어 있어도 되며, 괄호 안은 허용하는 거리입니다.
- `tool-retry-failure`는 기록을 만드는 쪽이 재시도를 `retry:`로 시작하는 단계로 명시해야 걸리는 엄격한
  패턴입니다. 그런 표시가 없는 기록에서는 `tool-repeat-failure-heuristic`처럼 이름에 heuristic이 붙은
  추정 패턴을 씁니다. 추정 패턴은 "재시도였다"는 증거가 아니라 조사할 후보입니다.
- `review-exportable`은 결과를 검토 후보로 내보낼 수 있는 패턴입니다(정리의 "다음 단계" 참고).

`tool-failure`는 실패한 TOOL step을 찾습니다. 파일 대신 디렉터리를 넘기면 그 안의 `*.json`
artifact를 모두 읽습니다.

In [20]:
tg("query", "tool-failure", TRACES)

$ tracegraph query tool-failure traces


tool-failure@v1: {kind=TOOL, status=error}  (over 2 trace(s))

A: call_tool

1 match(es) across 1 trace(s)
[exit code 0]


> **해석**
> - *말해 주는 것*: 읽은 trace 2개 중 A에서 패턴이 1건 나왔습니다. 첫 줄의 `(over 2 trace(s))`가
>   검색 대상의 수입니다.
> - *말해 주지 않는 것*: 비율이나 추세는 직접 계산해야 합니다(실무 응용 2). 또 매치가 없다는 결과는
>   실패가 제대로 기록됐을 때만 의미가 있습니다. 실습 3 끝에서 본 것처럼 실패 기록 방식이 맞지 않으면
>   실패가 있어도 매치는 0입니다.
> - *다음에 할 일*: 매치된 trace마다 `--explain`으로 원인 경로를 보고, 여러 사례에 공통된 앞 단계가
>   있는지 찾습니다.

매치가 없으면 exit code **1**입니다. 이것도 오류가 아니라 "없다"는 답입니다. 지금 두 trace에는
timeout으로 실패한 step이 없습니다.

In [21]:
tg("query", "timeout", TRACES, expect=1)

$ tracegraph query timeout traces


timeout@v1: {error~=timeout, status=error}  (over 2 trace(s))

no matches
[exit code 1]


**CI에서 쓰는 예.** "테스트 시나리오를 돌린 실행에서 tool 실패가 하나도 없어야 한다"는 규칙은
이렇게 씁니다. 매치가 있으면(exit code 0) CI를 실패시키고, 2(사용법 오류)는 그대로 오류로 둡니다.
GitHub Actions의 bash는 `-e` 옵션으로 실행되어 exit code 1에서 바로 멈추므로, exit code를 변수로
받아 둡니다.

```bash
rc=0
tracegraph query tool-failure artifacts/ || rc=$?
case $rc in
  0) echo "tool failure found"; exit 1 ;;
  1) echo "no tool failure" ;;
  *) exit "$rc" ;;
esac
```

`--explain`을 붙이면 매치마다 원인 경로까지 함께 보여 줍니다. 매치를 찾은 다음 바로 원인을 보고 싶을
때 씁니다. 매치가 많으면 `--limit`으로 개수를 줄일 수 있습니다.

In [22]:
tg("query", "tool-failure", "--explain", TRACES)

$ tracegraph query tool-failure --explain traces


tool-failure@v1: {kind=TOOL, status=error}  (over 2 trace(s))

A: call_tool
    ← plan (step 1)
    ← CHAIN (step 0)
    ← CHAIN (step -1)

1 match(es) across 1 trace(s)
[exit code 0]


---

## 실무 응용 1: 변경 검증 — 재시도 로직을 추가했다

지금까지는 명령을 하나씩 봤습니다. 실무에서는 이 명령들을 묶어서 "내 변경이 의도대로 동작하는가"를
확인합니다.

**상황.** 일시적인 tool 오류를 줄이려고, tool이 실패하면 한 번 더 시도하는 v2를 만들었습니다. 배포 전에
두 가지를 확인하고 싶습니다.

1. **회귀가 없는가?** 정상 입력에서는 v1과 같은 경로로 실행돼야 합니다.
2. **의도대로 동작하는가?** 실패 입력에서는 재시도한 뒤 오류 처리로 가야 합니다.

아래가 v2 그래프입니다. v1(`tiny_agent.py`)과 같은 노드에 `prepare_retry`가 추가됐고, `call_tool`이
실패하면 시도 횟수가 2회가 될 때까지 다시 시도합니다.

In [23]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class StateV2(TypedDict):
    input: str
    error: str | None
    result: str | None
    attempts: int
    log: Annotated[list[str], operator.add]


def plan(state):
    return {"log": ["planned"]}


def call_tool(state):
    attempt = state["attempts"] + 1
    if "boom" in state["input"]:
        return {"error": f"tool failed (attempt {attempt})", "attempts": attempt, "log": ["tool_error"]}
    return {"error": None, "result": "tool ok", "attempts": attempt, "log": ["tool_ok"]}


def prepare_retry(state):
    # 재시도 전에 이전 오류를 지웁니다. 아래 "해석"에서 이유를 설명합니다.
    return {"error": None, "log": ["retrying"]}


def handle_error(state):
    return {"result": "recovered", "log": ["handled"]}


def respond(state):
    return {"result": state.get("result") or "done", "log": ["responded"]}


def route(state):
    if not state.get("error"):
        return "respond"
    return "prepare_retry" if state["attempts"] < 2 else "handle_error"


def build_v2(checkpointer):
    g = StateGraph(StateV2)
    for name, node in [("plan", plan), ("call_tool", call_tool), ("prepare_retry", prepare_retry),
                       ("handle_error", handle_error), ("respond", respond)]:
        g.add_node(name, node)
    g.add_edge(START, "plan")
    g.add_edge("plan", "call_tool")
    g.add_conditional_edges("call_tool", route, ["prepare_retry", "handle_error", "respond"])
    g.add_edge("prepare_retry", "call_tool")
    g.add_edge("handle_error", "respond")
    g.add_edge("respond", END)
    return g.compile(checkpointer=checkpointer)


CHANGE = WORK / "change"
CHANGE.mkdir()
with SqliteSaver.from_conn_string(str(TRACE_DB)) as saver:
    app_v2 = build_v2(saver)
    for thread, text in [("A-v2", "boom"), ("B-v2", "hello")]:
        state = {"input": text, "error": None, "result": None, "attempts": 0, "log": []}
        print(f"thread {thread}:", app_v2.invoke(state, {"configurable": {"thread_id": thread}}))

A_V2, B_V2 = CHANGE / "A-v2.json", CHANGE / "B-v2.json"
tg("ingest", "--sqlite", TRACE_DB, "--thread", "A-v2", "--out", A_V2, quiet=True)
tg("ingest", "--sqlite", TRACE_DB, "--thread", "B-v2", "--out", B_V2, quiet=True)

thread A-v2: {'input': 'boom', 'error': 'tool failed (attempt 2)', 'result': 'recovered', 'attempts': 2, 'log': ['planned', 'tool_error', 'retrying', 'tool_error', 'handled', 'responded']}
thread B-v2: {'input': 'hello', 'error': None, 'result': 'tool ok', 'attempts': 1, 'log': ['planned', 'tool_ok', 'responded']}
$ tracegraph ingest --sqlite trace.db --thread A-v2 --out change/A-v2.json


[exit code 0]
$ tracegraph ingest --sqlite trace.db --thread B-v2 --out change/B-v2.json


[exit code 0]


**확인 1: 정상 입력에서 회귀가 없는가?** 같은 입력 `"hello"`로 돌린 v1(B)과 v2를 비교합니다.

In [24]:
tg("diff", B_JSON, B_V2)

$ tracegraph diff traces/B.json change/B-v2.json


Derived TREE_PARENT comparison; lossy steps: A=0, B=0
IDENTICAL — the two runs are structurally isomorphic.
[exit code 0]


`IDENTICAL`, exit code 0입니다. 두 실행의 step id는 전부 다르지만(실행마다 새로 생기므로) 구조는
같습니다. 정상 입력에서 v2는 v1과 같은 실행 트리(구조와 step 이름)를 만들었습니다. `diff`는 상태나
출력은 비교하지 않으므로, 상태 변화는 `analyze --baseline`으로, 응답 내용은 별도의 평가로 확인합니다.

**확인 2: 실패 입력에서 의도대로 재시도하는가?**

In [25]:
tg("inspect", A_V2)

$ tracegraph inspect change/A-v2.json


A-v2 (langgraph)
└── -1: CHAIN
    └── 0: CHAIN
        └── 1: plan
            └── 2: call_tool  ✗ tool failed (attempt 1)
                └── 3: prepare_retry
                    └── 4: call_tool  ✗ tool failed (attempt 2)
                        └── 5: handle_error
                            └── 6: respond

8 steps · 2 tool · 2 error · 0 lossy-projection
[exit code 0]


In [26]:
tg("analyze", A_V2, "--baseline", A_JSON)

$ tracegraph analyze change/A-v2.json --baseline traces/A.json


A-v2  8 steps · 2 error

What failed — candidates to investigate
  ✗ call_tool (TOOL, step 2)
    CHAIN → CHAIN [checkpoint_parent]
    CHAIN → plan [checkpoint_parent]
    plan → call_tool [checkpoint_parent]
  propagated/context errors: call_tool

Repeated behavior
  • plan-then-tool-failure@v1: plan → call_tool
  • tool-failure@v1: call_tool
  • tool-failure@v1: call_tool
  • error@v1: call_tool
  • error@v1: call_tool

Telemetry
  wall=unavailable · tokens=unavailable · cost=unavailable

Compared with baseline
  topology: changed
  • handle_error: ok → missing
  • respond: ok → missing
  • prepare_retry: missing → ok
  • call_tool: missing → error
  • handle_error: missing → ok
  • respond: missing → ok
  pattern deltas: error=+1, tool-failure=+1
  telemetry delta: wall=unavailable · tokens=unavailable · cost=unavailable

Data fidelity / privacy
  declared_dag · links_preserved=None · privacy=safe-v1
  ⚠ current: LangGraph failure capture covers the configured error channel and tas

> **해석**
> - *말해 주는 것*: v2는 의도대로 `call_tool` → `prepare_retry` → `call_tool` → `handle_error` 순서로
>   움직였습니다. 그런데 두 번째 시도도 실패했습니다. `pattern deltas`의 `tool-failure=+1`은 "v1보다
>   tool 실패가 한 번 더 기록됐다"는 뜻입니다.
> - *실패 후보가 하나뿐인 이유*: 두 번째 실패는 인과 경로상 앞에 이미 실패(첫 번째 시도)가 있으므로
>   `propagated/context errors`로 따로 분류됩니다. 먼저 조사할 곳은 앞쪽의 실패라는 뜻입니다.
> - *말해 주지 않는 것*: 이 tool의 실패가 일시적인지 아닌지는 알려 주지 않습니다. 다만 이 예제처럼 같은
>   입력에서 매번 실패한다면 재시도는 호출 비용과 지연만 늘립니다.
> - *다음에 할 일*: 실제 운영 기록에서 재시도가 성공으로 이어진 비율을 확인한 뒤 재시도 정책을
>   정합니다(실무 응용 2의 집계 방식을 쓸 수 있습니다).
>
> **기록 측면의 팁**: tracegraph는 `error` 키에 값이 **처음 나타난** step을 실패로 봅니다. state의
> 값은 다음 단계로 이어지기 때문입니다. 그래서 이전 오류가 남아 있는 채로 다시 실패하면 두 번째
> 실패가 따로 기록되지 않습니다. v2의 `prepare_retry`처럼 재시도 전에 오류 키를 비우면 각 시도의
> 실패가 각각 기록됩니다.

**재시도 패턴 찾기.** 재시도 후 또 실패한 경우를 여러 실행에서 찾을 때는 `query`를 씁니다. 패턴이
두 종류라는 점에 주의하세요.

In [27]:
tg("query", "tool-repeat-failure-heuristic", "--explain", CHANGE)

$ tracegraph query tool-repeat-failure-heuristic --explain change


tool-repeat-failure-heuristic@v1: {kind=TOOL} →[gap 2..]→ {kind=TOOL, status=error, name=#0}  (over 2 trace(s))

A-v2: call_tool → call_tool
    ← prepare_retry (step 3)
    ← call_tool (step 2)
    ← plan (step 1)
    ← CHAIN (step 0)
    ← CHAIN (step -1)

1 match(es) across 1 trace(s)
[exit code 0]


In [28]:
tg("query", "tool-retry-failure", CHANGE, expect=1)

$ tracegraph query tool-retry-failure change


tool-retry-failure@v2: {kind=TOOL} → {name^=retry:, kind=CHAIN} → {kind=TOOL, status=error, name=#0}  (over 2 trace(s))

no matches
[exit code 1]


- `tool-repeat-failure-heuristic`은 "같은 tool이 몇 단계 뒤에 다시 나와 실패했다"는 모양만 보고
  매치합니다. v2의 재시도가 잡혔지만, 이것이 재시도였다는 **증거**는 아니고 조사할 후보입니다.
- `tool-retry-failure`는 기록을 만드는 쪽이 둘 사이에 `retry:`로 시작하는 단계를 명시해야 매치합니다.
  v2는 그런 표시를 남기지 않으므로 매치가 없습니다(exit code 1). 명시적인 재시도 표시를 남기는
  producer(예: SyncMill)의 기록에서는 이 엄격한 패턴을 쓰고, 그 결과만 검토 후보로 내보낼 수 있습니다.

**테스트 코드에 넣기.** 같은 확인을 agent 테스트에 넣으면 변경할 때마다 자동으로 검사됩니다. 두 가지를
나눠서 검사합니다.

- **회귀 없음**은 `diff`의 exit code로 확인합니다. 0과 1만 정상적인 답으로 보고, 2(사용법 오류)는 테스트
  오류로 처리합니다.
- **의도한 경로**는 `diff`로 확인할 수 없습니다. "구조가 다르다"는 재시도가 아니라 노드 누락이나 엉뚱한
  분기여도 참이기 때문입니다. 그래서 artifact에서 step 순서와 상태를 직접 확인합니다.

In [29]:
def same_structure(baseline: Path, candidate: Path) -> bool:
    """두 artifact 의 실행 구조가 같으면 True. tracegraph diff 의 exit code 를 그대로 씁니다."""
    rc = subprocess.run([str(TRACEGRAPH), "diff", str(baseline), str(candidate)], capture_output=True).returncode
    if rc not in (0, 1):
        raise RuntimeError(f"tracegraph diff failed with exit code {rc}")
    return rc == 0


def step_path(artifact_path: Path) -> list[tuple[str, str]]:
    """이름 있는 step 을 실행 순서대로 (이름, 상태) 로 돌려줍니다."""
    steps = json.loads(artifact_path.read_text())["trace"]["steps"]
    return [(s["name"], s["status"]) for s in sorted(steps, key=lambda s: s["seq"]) if s["name"]]


# pytest 라면 test_*.py 함수 안에 두면 됩니다.
assert same_structure(B_JSON, B_V2), "정상 입력의 실행 구조가 바뀌었습니다"
assert step_path(A_V2) == [
    ("plan", "ok"),
    ("call_tool", "error"),  # 첫 시도 실패
    ("prepare_retry", "ok"),
    ("call_tool", "error"),  # 재시도도 실패
    ("handle_error", "ok"),  # 두 번 실패한 뒤 오류 처리로 복구
    ("respond", "ok"),
], step_path(A_V2)
print("v2 검증 통과: 정상 입력은 v1과 같은 실행 트리, 실패 입력은 2회 시도 후 오류 처리")

v2 검증 통과: 정상 입력은 v1과 같은 실행 트리, 실패 입력은 2회 시도 후 오류 처리


## 실무 응용 2: 여러 실행 모아 보기 — 실패 현황 집계

**상황.** 고객 문의 8건을 agent가 처리했습니다. 사용자에게는 모두 답이 나갔습니다. 알고 싶은 것은
다음 세 가지입니다.

1. 몇 건에서 실패가 있었나? (비율)
2. 어느 단계가 주로 실패하나?
3. 사용자는 몰랐지만 내부에서 실패한, 즉 복구된 실패는 몇 건인가?

`query`는 어디서 매치됐는지 알려 주지만 전체 중 비율은 알려 주지 않습니다. 실행마다 `analyze
--json-out`으로 보고서를 만들고 그 JSON을 모으면 이런 집계를 할 수 있습니다. 보고서 형식은 버전 관리되는
계약이라 집계 스크립트를 안정적으로 짤 수 있습니다.

먼저 v1 agent로 8건을 처리합니다. 입력에 `boom`이 들어 있으면 tool이 실패하도록 만든 예제입니다.

In [30]:
INPUTS = ["주문 조회", "환불 요청 boom", "배송 문의", "주소 변경", "결제 오류 boom", "쿠폰 적용", "반품 boom", "재고 확인"]
BATCH, REPORTS = WORK / "batch", WORK / "reports"
BATCH.mkdir()
REPORTS.mkdir()

final_result = {}
with SqliteSaver.from_conn_string(str(TRACE_DB)) as saver:
    for i, text in enumerate(INPUTS):
        thread = f"run-{i:02d}"
        final_result[thread] = tiny_agent.run(saver, thread, text)["result"]

for thread in final_result:
    tg("ingest", "--sqlite", TRACE_DB, "--thread", thread, "--out", BATCH / f"{thread}.json", quiet=True)
    tg("analyze", BATCH / f"{thread}.json", "--json-out", REPORTS / f"{thread}.json", quiet=True)

$ tracegraph ingest --sqlite trace.db --thread run-00 --out batch/run-00.json


[exit code 0]
$ tracegraph analyze batch/run-00.json --json-out reports/run-00.json


[exit code 0]
$ tracegraph ingest --sqlite trace.db --thread run-01 --out batch/run-01.json


[exit code 0]
$ tracegraph analyze batch/run-01.json --json-out reports/run-01.json


[exit code 0]
$ tracegraph ingest --sqlite trace.db --thread run-02 --out batch/run-02.json


...

[exit code 0]
$ tracegraph ingest --sqlite trace.db --thread run-05 --out batch/run-05.json


[exit code 0]
$ tracegraph analyze batch/run-05.json --json-out reports/run-05.json


[exit code 0]
$ tracegraph ingest --sqlite trace.db --thread run-06 --out batch/run-06.json
[exit code 0]
$ tracegraph analyze batch/run-06.json --json-out reports/run-06.json


[exit code 0]
$ tracegraph ingest --sqlite trace.db --thread run-07 --out batch/run-07.json
[exit code 0]
$ tracegraph analyze batch/run-07.json --json-out reports/run-07.json


[exit code 0]


어느 실행에서 tool 실패가 났는지는 `query` 한 번으로 볼 수 있습니다.

In [31]:
tg("query", "tool-failure", BATCH)

$ tracegraph query tool-failure batch
tool-failure@v1: {kind=TOOL, status=error}  (over 8 trace(s))

run-01: call_tool
run-04: call_tool
run-06: call_tool

3 match(es) across 3 trace(s)
[exit code 0]


보고서를 모아 집계합니다. 보고서에는 사용자 입력이 들어 있지 않으므로, 입력 문구는 우리가 가진 실행
목록(`INPUTS`)과 trace id로 이어 붙입니다.

실패 step을 셀 때는 `primary_failures`(먼저 볼 실패)와 `propagated_failures`(그 하류의 실패)를 합칩니다.
`primary_failures`만 세면 실무 응용 1의 v2처럼 한 실행 안에서 여러 번 실패한 경우를 덜 세게 됩니다.
두 목록을 합친 수는 보고서의 `error_count`와 같아야 합니다.

In [32]:
import unicodedata
from collections import Counter


def pad(text, width):
    """한글처럼 두 칸을 차지하는 글자를 고려해 칸을 맞춥니다."""
    used = sum(2 if unicodedata.east_asian_width(ch) in "WF" else 1 for ch in str(text))
    return f"{text}{' ' * max(width - used, 0)}"


reports = {p.stem: json.loads(p.read_text()) for p in sorted(REPORTS.glob("*.json"))}

def failed_steps(report):
    """보고서의 모든 실패 step. 먼저 볼 실패(primary)와 그 하류의 실패(propagated)를 합칩니다."""
    steps = [f["step"] for f in report["primary_failures"]] + report["propagated_failures"]
    assert len(steps) == report["error_count"]
    return steps


print(pad("run", 8), pad("input", 16), pad("사용자 응답", 12), pad("기록된 상태", 12), "실패 step")
for i, (thread, report) in enumerate(reports.items()):
    failed = ", ".join(s["name"] for s in failed_steps(report)) or "-"
    print(pad(thread, 8), pad(INPUTS[i], 16), pad(final_result[thread], 12), pad(report["status"], 12), failed)

failed_runs = [t for t, r in reports.items() if r["error_count"]]
hidden = [t for t in failed_runs if final_result[t] == "recovered"]
failing_steps = Counter(s["name"] for r in reports.values() for s in failed_steps(r))
patterns = Counter(f"{p['pattern_id']}@v{p['pattern_version']}" for r in reports.values() for p in r["patterns"])

print()
print(f"실패가 기록된 실행 : {len(failed_runs)}/{len(reports)} ({len(failed_runs) / len(reports):.0%})")
print(f"그중 복구된 실패   : {len(hidden)}건 (사용자 응답은 'recovered')")
print("실패한 step       :", dict(failing_steps))
print("걸린 패턴         :", dict(patterns))

# 확인: 입력에 boom 이 들어간 3건만 실패했고, 모두 복구됐습니다.
assert failed_runs == hidden == [f"run-{i:02d}" for i, text in enumerate(INPUTS) if "boom" in text]

run      input            사용자 응답  기록된 상태  실패 step
run-00   주문 조회        tool ok      ok           -
run-01   환불 요청 boom   recovered    error        call_tool
run-02   배송 문의        tool ok      ok           -
run-03   주소 변경        tool ok      ok           -
run-04   결제 오류 boom   recovered    error        call_tool
run-05   쿠폰 적용        tool ok      ok           -
run-06   반품 boom        recovered    error        call_tool
run-07   재고 확인        tool ok      ok           -

실패가 기록된 실행 : 3/8 (38%)
그중 복구된 실패   : 3건 (사용자 응답은 'recovered')
실패한 step       : {'call_tool': 3}
걸린 패턴         : {'plan-then-tool-failure@v1': 3, 'tool-failure@v1': 3, 'error@v1': 3}


> **해석**
> - *말해 주는 것*: 8건 중 3건(38%)에서 tool 실패가 기록됐고, 3건 모두 오류 처리로 복구돼 사용자는
>   알아채지 못했습니다. 실패는 모두 `call_tool`에서 났습니다.
> - *말해 주지 않는 것*: 왜 이 3건이 실패했는지는 알려 주지 않습니다. 공통점을 찾는 것은 사람의
>   몫입니다. 이 예제에서는 세 입력 모두 `boom`을 포함한다는 것이 공통점입니다.
> - *다음에 할 일*: 공통점을 바탕으로 tool을 고친 뒤, 같은 입력으로 다시 돌려 실무 응용 1의 방식으로
>   변경을 검증합니다. 이 집계를 매일이나 매주 돌리면 실패율 추세를 볼 수 있습니다. 보고서에는 본문이
>   없어 대시보드에 올리기 쉽지만, trace id와 노드 이름은 남으므로 실습 3의 주의 사항을 먼저 확인하세요.

`analyze` 보고서의 `status`는 실행 결과가 아니라 **기록된 상태**입니다. 사용자 응답이 정상이어도
실패가 기록되면 `error`입니다. "사용자 응답"과 "기록된 상태"를 나란히 놓은 것이 이 집계의 핵심입니다.

---

## 실습 9 (심화): fan-in과 lossy-projection (OTLP)

> **왜 중요한가요?** 실제 agent에는 여러 결과를 하나로 합치는 단계가 흔합니다. 여러 문서를 검색해
> 답을 쓰는 RAG, 여러 하위 agent의 결과를 모으는 감독 agent가 그렇습니다. 이런 단계의 원인은 하나가
> 아닙니다(fan-in). 답변 품질 문제가 검색 결과 때문인데 트리에는 계획 단계만 부모로 보인다면, 엉뚱한
> 곳을 고치게 됩니다.

LangGraph 예제는 한 줄로 이어진 실행이라 raw graph와 트리가 같았습니다. `examples/otlp_agent_trace.py`는
OpenTelemetry로 계측한 agent가 내보낼 법한 OTLP trace를 만듭니다.

```
agent ─▶ plan ─┬─▶ retrieve_docs
               ├─▶ web_search   (✗ tool error)
               └─▶ synthesize   ◀── link ── retrieve_docs
```

`synthesize`의 원인은 그래프 부모인 `plan`과, span link로 연결된 `retrieve_docs` **두 개**입니다.

In [33]:
import otlp_agent_trace

OTLP_RAW, OTLP_JSON = RAW / "otlp_trace.json", TRACES / "otlp.json"
OTLP_RAW.write_text(json.dumps(otlp_agent_trace.sample_otlp_document(), indent=2))
tg("ingest-otlp", "--file", OTLP_RAW, "--out", OTLP_JSON)

$ tracegraph ingest-otlp --file raw/otlp_trace.json --out traces/otlp.json


ingested 5 spans → traces/otlp.json
[exit code 0]


In [34]:
tg("inspect", OTLP_JSON)

$ tracegraph inspect traces/otlp.json


agent-trace-1 (otlp)
└── 0: agent
    └── 1: plan
        ├── 2: retrieve_docs
        ├── 3: web_search  ✗ tool failed: upstream timeout
        └── 4: synthesize  ⚠ lossy-projection

5 steps · 1 tool · 1 error · 1 lossy-projection
[exit code 0]


트리에서는 부모를 하나만 그릴 수 있으므로 `synthesize`는 `plan` 아래에만 나타나고,
`retrieve_docs`에서 온 원인은 생략됩니다. 그래서 `⚠ lossy-projection`이 붙습니다. "이 트리는 이
step의 원인을 다 보여 주지 않는다"는 경고입니다. artifact에서도 확인할 수 있습니다.

In [35]:
otlp = json.loads(OTLP_JSON.read_text())["trace"]
names = {s["step_id"]: s["name"] for s in otlp["steps"]}
synthesize = next(s for s in otlp["steps"] if s["name"] == "synthesize")
causes = [e for e in otlp["edges"] if e["type"] == "CAUSED_BY" and e["src"] == synthesize["step_id"]]
parents = [e for e in otlp["edges"] if e["type"] == "TREE_PARENT" and e["src"] == synthesize["step_id"]]
print("projection_lossy:", synthesize["projection_lossy"])
print("CAUSED_BY  →", [(names[e["dst"]], e["origin"]) for e in causes])
print("TREE_PARENT →", [names[e["dst"]] for e in parents])

# 확인: raw graph 에는 원인이 둘, 트리에는 부모가 하나입니다.
assert synthesize["projection_lossy"] is True
assert sorted(names[e["dst"]] for e in causes) == ["plan", "retrieve_docs"]
assert len(parents) == 1

projection_lossy: True
CAUSED_BY  → [('plan', 'graph_parent'), ('retrieve_docs', 'span_link')]
TREE_PARENT → ['plan']


edge의 `src`는 결과, `dst`는 원인입니다. `origin`은 그 원인을 어떤 근거로 알았는지(`graph_parent`:
그래프 부모 관계, `span_link`: span link)를 기록합니다. 근거가 남아 있으므로 나중에 "이 인과관계는
어디서 왔나"를 되짚을 수 있습니다.

`explain`은 raw graph를 읽으므로 두 원인을 모두 보여 줍니다. `synthesize`의 step id는 OTLP span id
그대로인 `s4`입니다.

In [36]:
tg("explain", OTLP_JSON, synthesize["step_id"])

$ tracegraph explain traces/otlp.json s4


synthesize (step 4) [ok] 
  ← plan (step 1)
  ← retrieve_docs (step 2)
  ← agent (step 0)

Some steps had multiple real causes; the single-parent tree view is lossy for them — trust this raw chain, not the tree.
[exit code 0]


`synthesize`의 원인 목록에 그래프 부모 `plan`과 link로 연결된 `retrieve_docs`가 모두 나옵니다.
`inspect` 트리에서는 보이지 않던 원인입니다. 마지막 줄은 이 trace에 원인이 여러 개인 step이 있으니
트리가 아니라 이 원본 인과 경로(raw chain)를 믿으라는 안내입니다.

참고로 Phoenix에서 내려받은 기록에는 span link가 없어서 이런 추가 원인을 복원할 수 없습니다.
tracegraph는 그런 artifact에 `parent_only`를 표시해 "추가 원인이 빠졌을 수 있다"고 알립니다. link까지
보존해야 한다면 OpenTelemetry Collector로 Phoenix와 tracegraph에 동시에 보내는 예제
[`examples/otel-collector-phoenix-tracegraph.yaml`](otel-collector-phoenix-tracegraph.yaml)을 참고하세요.

이제 `traces/`에 OTLP trace가 추가됐으므로, 실습 8에서 매치가 없던 `timeout` 패턴이 `web_search`의
`upstream timeout` 실패에 걸립니다. 같은 명령의 exit code가 1에서 0으로 바뀌는 것을 확인하세요.
기록이 쌓일수록 같은 `query`가 새 사례를 찾아 주는 방식입니다.

In [37]:
tg("query", "timeout", TRACES)

$ tracegraph query timeout traces


timeout@v1: {error~=timeout, status=error}  (over 3 trace(s))

agent-trace-1: web_search

1 match(es) across 1 trace(s)
[exit code 0]


## 실습 10 (심화): 내보내기 (`export-incident-memo`)

> **언제 쓰나요?** 장애 회고를 팀 위키나 장기 기억 저장소에 남길 때입니다. 실행 기록에는 사용자
> 입력이나 내부 정보가 섞여 있을 수 있어서 그대로 공유하기 어렵습니다. 이 메모는 **구조만** 남깁니다.

`export-incident-memo`는 artifact를 읽어 사후 분석 메모(Markdown)를 씁니다. step 이름과 오류
메시지는 쓰지 않고, step은 `step_<번호>_<종류>` 별칭으로, 실행은 artifact의 SHA-256 digest로만
나타냅니다. digest가 있으므로 원본 artifact를 가진 사람은 메모가 어느 실행에 대한 것인지 정확히 맞춰
볼 수 있습니다.

In [38]:
MEMO = OUT / "incident.md"
tg("export-incident-memo", A_JSON, "-o", MEMO)
print()
print(MEMO.read_text())

$ tracegraph export-incident-memo traces/A.json -o out/incident.md


exported incident memo → out/incident.md
[exit code 0]

---
name: incident_bd832645139e
description: Tracegraph incident post-mortem for artifact sha256:bd832645139ee949...
type: incident
metadata:
  node_type: memory
  type: incident
  artifact_digest: sha256:bd832645139ee949af7fb993c356d770a38dd5e7a9dd986eb036171a37c71f43
  causal_fidelity: declared_dag
  links_preserved: null
---

# Incident Post-Mortem: `incident_bd832645139e`

## Summary
- **Trace Status**: `error`
- **Artifact Digest**: `sha256:bd832645139ee949af7fb993c356d770a38dd5e7a9dd986eb036171a37c71f43`
- **Causal Fidelity**: `declared_dag`
- **Links Preserved**: `null`
- **Step Count**: 6 (Errors: 1)

## Diagnostic Findings
### Detected Patterns
- **plan-then-tool-failure** (v1): steps `[step_3_chain, step_4_tool]`
- **tool-failure** (v1): steps `[step_4_tool]`
- **error** (v1): steps `[step_4_tool]`

### Failure Causality (Actual Recorded Predecessors)
- Failure: `step_4_tool` (status: error)
  - Direct recorded predecess

별칭 번호는 `seq`가 아니라 1부터 센 순서입니다. A의 여섯 step 중 네 번째인 `call_tool`이
`step_4_tool`입니다. 오류 문자열 `boom`은 메모 어디에도 없습니다.

포함 관계만 보여 주는 edge나 출처를 모르는 edge는 "caused by"가 아니라 "preceded by"로 적어서, 메모를
읽는 사람이 인과관계를 실제보다 강하게 받아들이지 않게 합니다.

## 정리

| 명령 | 답하는 질문 | exit code |
|---|---|---|
| `ingest`, `ingest-otlp` | 기록을 artifact로 바꾸기 | 0 = 성공 |
| `validate` | artifact가 온전한가? | 0 = 모두 유효 |
| `analyze` | 무엇이 실패했고, 기준 실행과 무엇이 다른가? | 0 = 분석 성공 (차이가 있어도 0) |
| `inspect` | 실행 흐름이 어떻게 생겼나? | 0 |
| `explain` | 이 step의 기록된 원인은 무엇인가? | 0 |
| `diff` | 두 실행의 구조가 같은가? | 0 = 같음, **1 = 다름** |
| `query` | 이 패턴이 여러 trace 중 어디에 있나? | 0 = 있음, **1 = 없음** |
| `export-incident-memo` | 공유 가능한 사후 분석 메모 만들기 | 0 = 성공 |

잘못된 인자나 없는 파일처럼 명령을 실행할 수 없는 경우는 exit code 2입니다.

### 내 프로젝트에 적용하는 순서

1. **기록 경로 고르기.** `SqliteSaver`를 쓰면 `ingest`, OpenTelemetry로 계측했다면 `ingest-otlp`,
   Phoenix를 쓰고 있다면 `phoenix diagnose`부터 시작합니다.
2. **실패가 잡히는지 확인하기.** 실패가 확실한 실행 하나로 `analyze`를 돌려 그 실패가 보이는지
   확인합니다. 안 보이면 `--error-channel`과 tool 노드 이름을 점검합니다(실습 3).
3. **기준 실행 저장하기.** 대표 입력마다 정상 실행의 artifact를 baseline으로 보관합니다.
4. **변경 후 비교하기.** 프롬프트·모델·코드를 바꾼 뒤 같은 입력으로 돌려 `analyze --baseline`과
   `diff`로 확인합니다(실무 응용 1).
5. **CI와 테스트에 연결하기.** 구조가 고정돼야 하는 흐름은 `diff`, 나오면 안 되는 패턴은 `query`의
   exit code로 막습니다(실습 7, 8, 실무 응용 1의 `same_structure`).
6. **운영 기록 살피기.** 쌓인 artifact에 주기적으로 `query`를 돌려 반복되는 실패 패턴을 찾고, 보고서를
   모아 실패율을 집계하고(실무 응용 2), 공유가 필요한 사례는 `export-incident-memo`로 남깁니다.

### 다음 단계

- **Phoenix**: Phoenix CLI `px`(1.0.4 이상)와 인증을 설정했다면 trace id 없이도 최근 실패를 진단할
  수 있습니다. tracegraph는 `px`의 읽기 전용 명령만 쓰고 원문 응답을 저장하지 않습니다. 이 노트북에서는
  실행하지 않습니다.

  ```bash
  tracegraph phoenix doctor
  tracegraph phoenix diagnose
  tracegraph phoenix diagnose <failed-trace-id> --baseline <successful-trace-id>
  ```

- **검토 후보 내보내기**: `export-review-candidates`는 `review-exportable` 패턴의 매치를 검토 대기열에
  넣을 수 있는 최소한의 JSON으로 내보냅니다. 실행에 `run_id`가 있고 tool 이름이 `server::tool`
  형태여야 합니다.
- **LadybugDB 백엔드**: `uv sync --extra cypher` 후 `query`, `explain`에 `--backend ladybug`를 붙이면
  Cypher 가속기로 같은 분석을 합니다. 결과의 의미는 기본 백엔드와 같습니다.
- 자세한 옵션과 개념은 [`docs/USAGE_KO.md`](../docs/USAGE_KO.md)와 [`README.md`](../README.md)를
  보세요.

실습 파일은 아래 셀로 지울 수 있습니다.

In [39]:
shutil.rmtree(WORK)
print("removed", WORK)

removed /var/folders/4n/mt3km5rs1bn_b0w_d6hvlmhc0000gn/T/tracegraph-tutorial-siaei0kp
